In [40]:
# sqlite3 is the standard Python library for SQLite databases
# pandas is used for reading CSV files and handling tabular data
import sqlite3
import pandas as pd

In [41]:
# Load the CSV into a DataFrame
df = pd.read_csv("train.csv")
df.head()

#Create SQLite Database and Table
conn = sqlite3.connect("titanic.db")

df.to_sql(
    name="titanic",
    con=conn,
    if_exists="replace",
    index=False
)

891

In [42]:
#Verify Table Exists
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
cursor.fetchall()


[('titanic',)]

In [43]:
#Row count
query = """
SELECT COUNT(*) AS total_rows
FROM titanic;
"""

pd.read_sql(query, conn)


,total_rows
0,891


In [44]:
# Calculate survival rate grouped by passenger class and gender
query = """
WITH survival_stats AS (
    SELECT
        Pclass,
        Sex,
        COUNT(*) AS total_passengers,
        SUM(Survived) AS survived_passengers
    FROM titanic
    GROUP BY Pclass, Sex
)
SELECT
    Pclass,
    Sex,
    ROUND(1.0 * survived_passengers / total_passengers, 3) AS survival_rate
FROM survival_stats
ORDER BY Pclass, Sex;
"""

pd.read_sql(query, conn)


,Pclass,Sex,survival_rate
0,1,female,0.968
1,1,male,0.369
2,2,female,0.921
3,2,male,0.157
4,3,female,0.500
5,3,male,0.135


In [45]:
# This query analyzes passenger survival by age category.
# The objective is to transform a continuous variable (Age)
# into discrete groups and then aggregate survival outcomes.

query = """
-- Use a Common Table Expression (CTE) to perform feature engineering
WITH age_groups AS (

    -- Categorize passengers into age groups using a CASE expression
    SELECT
        CASE
            -- Passengers younger than 18 are classified as 'Child'
            WHEN Age < 18 THEN 'Child'

            -- Passengers between 18 and 60 (inclusive) are 'Adult'
            WHEN Age BETWEEN 18 AND 60 THEN 'Adult'

            -- Passengers older than 60 are classified as 'Senior'
            ELSE 'Senior'
        END AS age_group,

        -- Survived is a binary variable (1 = survived, 0 = did not survive)
        -- Keeping this column allows us to count survivors via aggregation
        Survived

    -- Exclude records with missing age values to avoid incorrect grouping
    FROM titanic
    WHERE Age IS NOT NULL
)

-- Aggregate survival statistics by age group
SELECT
    -- Age category produced in the CTE
    age_group,

    -- Total number of passengers in each age group
    COUNT(*) AS total_passengers,

    -- Total number of survivors in each age group
    -- SUM works because Survived is encoded as 0 or 1
    SUM(Survived) AS survived_passengers

-- Group results by the derived age category
FROM age_groups
GROUP BY age_group;
"""

# Execute the SQL query against the SQLite database
# and return the result as a Pandas DataFrame
pd.read_sql(query, conn)


,age_group,total_passengers,survived_passengers
0,Adult,579,224
1,Child,113,61
2,Senior,22,5


In [46]:
# Compute the average ticket fare for each passenger class
query = """
SELECT
    Pclass,
    ROUND(AVG(Fare), 2) AS average_fare
FROM titanic
GROUP BY Pclass;
"""

pd.read_sql(query, conn)


,Pclass,average_fare
0,1,84.15
1,2,20.66
2,3,13.68


In [47]:
# Analyze survival based on family size (including the passenger)
query = """
WITH family_size AS (
    SELECT
        Survived,
        (SibSp + Parch + 1) AS family_count
    FROM titanic
)
SELECT
    family_count,
    COUNT(*) AS total_passengers,
    SUM(Survived) AS survived_passengers
FROM family_size
GROUP BY family_count
ORDER BY family_count;
"""

pd.read_sql(query, conn)


,family_count,total_passengers,survived_passengers
0,1,537,163
1,2,161,89
2,3,102,59
3,4,29,21
4,5,15,3
5,6,22,3
6,7,12,4
7,8,6,0
8,11,7,0


In [48]:
# Calculate survival rates by embarkation port
query = """
WITH embark_stats AS (
    SELECT
        Embarked,
        COUNT(*) AS total_passengers,
        SUM(Survived) AS survived_passengers
    FROM titanic
    WHERE Embarked IS NOT NULL
    GROUP BY Embarked
)
SELECT
    Embarked,
    ROUND(1.0 * survived_passengers / total_passengers, 3) AS survival_rate
FROM embark_stats
ORDER BY survival_rate DESC;
"""

pd.read_sql(query, conn)


,Embarked,survival_rate
0,C,0.554
1,Q,0.390
2,S,0.337


In [49]:
# Identify passengers who paid more than the average fare
query = """
WITH avg_fare AS (
    SELECT AVG(Fare) AS avg_ticket_fare
    FROM titanic
)
SELECT
    PassengerId,
    Pclass,
    Fare
FROM titanic, avg_fare
WHERE Fare > avg_ticket_fare
ORDER BY Fare DESC;
"""

pd.read_sql(query, conn)


,PassengerId,Pclass,Fare
0,259,1,512.3292
1,680,1,512.3292
2,738,1,512.3292
3,28,1,263.0000
4,89,1,263.0000
...,...,...,...
206,597,2,33.0000
207,721,2,33.0000
208,849,2,33.0000
209,417,2,32.5000


## WITH CTE

In [50]:
# This query analyzes passenger survival by age category
# and calculates the survival rate for each group.
# Survival rate is defined as:
#   number of survivors / total passengers in the group

query = """
-- Step 1: Create a CTE (Common Table Expression) to engineer age-based categories
WITH age_groups AS (

    -- Convert the numeric Age column into categorical age groups
    SELECT
        CASE
            -- Children: younger than 18 years old
            WHEN Age < 18 THEN 'Child'

            -- Adults: between 18 and 60 years old (inclusive)
            WHEN Age BETWEEN 18 AND 60 THEN 'Adult'

            -- Seniors: older than 60 years old
            ELSE 'Senior'
        END AS age_group,

        -- Binary survival indicator (1 = survived, 0 = did not survive)
        Survived

    -- Remove rows with missing Age values to ensure valid grouping
    FROM titanic
    WHERE Age IS NOT NULL
)

-- Step 2: Aggregate survival statistics by age group
SELECT
    -- Age category label
    age_group,

    -- Total number of passengers in this age group
    COUNT(*) AS total_passengers,

    -- Total number of survivors in this age group
    -- SUM works because Survived is encoded as 0 or 1
    SUM(Survived) AS survived_passengers,

    -- Survival rate calculation:
    -- Cast to floating point (1.0 *) to avoid integer division
    -- Rounded to 3 decimal places for readability
    ROUND(1.0 * SUM(Survived) / COUNT(*), 3) AS survival_rate

-- Use the engineered age groups from the CTE
FROM age_groups

-- Aggregate results by age category
GROUP BY age_group;
"""

# Execute the SQL query and return the result as a Pandas DataFrame
pd.read_sql(query, conn)


,age_group,total_passengers,survived_passengers,survival_rate
0,Adult,579,224,0.387
1,Child,113,61,0.540
2,Senior,22,5,0.227


## WITHOUT CTE

In [38]:
# This query analyzes passenger survival by age category
# and calculates the survival rate for each group,
# WITHOUT using a Common Table Expression (CTE).
# All feature engineering is done inside an inline subquery.

query = """
-- Step 1: Perform age categorization inside a subquery
SELECT
    -- Age category derived from the CASE expression
    age_group,

    -- Total number of passengers in each age group
    COUNT(*) AS total_passengers,

    -- Total number of survivors in each age group
    -- SUM works because Survived is a binary variable (0 or 1)
    SUM(Survived) AS survived_passengers,

    -- Survival rate calculation:
    -- Multiply by 1.0 to force floating-point division
    -- Round to 3 decimal places for readability
    ROUND(1.0 * SUM(Survived) / COUNT(*), 3) AS survival_rate

-- The subquery generates age_group values on the fly
FROM (
    SELECT
        CASE
            -- Children: younger than 18
            WHEN Age < 18 THEN 'Child'

            -- Adults: between 18 and 60 (inclusive)
            WHEN Age BETWEEN 18 AND 60 THEN 'Adult'

            -- Seniors: older than 60
            ELSE 'Senior'
        END AS age_group,

        -- Binary survival indicator
        Survived

    -- Filter out rows with missing Age values
    FROM titanic
    WHERE Age IS NOT NULL
)

-- Aggregate by the derived age group
GROUP BY age_group;
"""

# Execute the SQL query and return the result as a Pandas DataFrame
pd.read_sql(query, conn)


,age_group,total_passengers,survived_passengers,survival_rate
0,Adult,579,224,0.387
1,Child,113,61,0.540
2,Senior,22,5,0.227


# Simpler Queries

In [52]:
# Count the total number of passengers in the Titanic dataset
# COUNT(*) counts every row in the table

query = """
SELECT COUNT(*) AS total_passengers
FROM titanic;
"""

pd.read_sql(query, conn)


,total_passengers
0,891


In [53]:
# Count how many passengers belong to each gender
# GROUP BY groups rows with the same value in the Sex column

query = """
SELECT
    Sex,
    COUNT(*) AS passenger_count
FROM titanic
GROUP BY Sex;
"""

pd.read_sql(query, conn)


,Sex,passenger_count
0,female,314
1,male,577


In [56]:
# Calculate the average age of passengers
# NULL age values are ignored automatically by AVG

query = """
SELECT
    ROUND(AVG(Age), 1) AS average_age
FROM titanic;
"""

pd.read_sql(query, conn)


,average_age
0,29.7


In [57]:
# Survived is a binary column (1 = survived, 0 = did not survive)
# SUM(Survived) counts survivors
# COUNT(*) - SUM(Survived) gives non-survivors

query = """
SELECT
    SUM(Survived) AS survived_passengers,
    COUNT(*) - SUM(Survived) AS non_survivors
FROM titanic;
"""

pd.read_sql(query, conn)


,survived_passengers,non_survivors
0,342,549


In [66]:
# Calculate the overall survival rate
# Multiplying by 1.0 avoids integer division

query = """
SELECT
    ROUND(1.0 * SUM(Survived) / COUNT(*), 4) AS survival_rate
FROM titanic;
"""

pd.read_sql(query, conn)


,survival_rate
0,0.3838


In [67]:
# Calculate the average ticket fare for each passenger class
# Shows economic differences between classes

query = """
SELECT
    Pclass,
    ROUND(AVG(Fare), 2) AS average_fare
FROM titanic
GROUP BY Pclass;
"""

pd.read_sql(query, conn)


,Pclass,average_fare
0,1,84.15
1,2,20.66
2,3,13.68


In [68]:
# Count total passengers and survivors for each class
# Combines COUNT and SUM with GROUP BY

query = """
SELECT
    Pclass,
    COUNT(*) AS total_passengers,
    SUM(Survived) AS survived_passengers
FROM titanic
GROUP BY Pclass;
"""

pd.read_sql(query, conn)


,Pclass,total_passengers,survived_passengers
0,1,216,136
1,2,184,87
2,3,491,119


In [69]:
# Count total passengers and survivors for each passenger class
# Also calculate the survival rate per class
# Survival rate = survived_passengers / total_passengers

query = """
SELECT
    Pclass,

    -- Total number of passengers in each class
    COUNT(*) AS total_passengers,

    -- Number of survivors in each class (Survived is 0 or 1)
    SUM(Survived) AS survived_passengers,

    -- Survival rate calculation
    -- Multiply by 1.0 to avoid integer division
    -- Rounded to 3 decimal places for readability
    ROUND(1.0 * SUM(Survived) / COUNT(*), 3) AS survival_rate

FROM titanic
GROUP BY Pclass;
"""

pd.read_sql(query, conn)


,Pclass,total_passengers,survived_passengers,survival_rate
0,1,216,136,0.630
1,2,184,87,0.473
2,3,491,119,0.242


In [71]:
# Close the SQLite connection when finished
conn.close()
